# CANVASAI : AI Image Enhancer and Outpainter
An AI-powered image enhancement and outpainting tool. Upload any photo — CanvasAI either sharpens and upscales it using Real-ESRGAN, or extends its borders with contextually generated content using Stable Diffusion inpainting.

# Install all Libraries

* gradio       → the UI framework + built-in public URL 
* diffusers    → runs Stable Diffusion inpainting model
* transformers → core Hugging Face library, used by BLIP caption model
* accelerate   → helps models load faster and use GPU memory efficiently
* pillow       → PIL: open, resize, crop, paste, save images
* numpy        → numerical operations — we use it to create the mask (black/white image)             a mask is just a 2D array of 0s and 255s — numpy is perfect for this
* basicsr      → base library that Real-ESRGAN is built on top of
* facexlib     → face detection library, required by Real-ESRGAN internally
* gfpgan       → face enhancement library, also required by Real-ESRGAN internally

In [1]:
!pip install -q basicsr facexlib gfpgan
!pip install -q git+https://github.com/xinntao/Real-ESRGAN.git
!pip install -q gradio diffusers transformers accelerate pillow numpy
print("all libraries installed")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.5/172.5 kB 3.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 3.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 8.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.2/52.2 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 85.6 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 21.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incom

In [2]:
import torchvision.transforms.functional as F
import sys
sys.modules["torchvision.transforms.functional_tensor"] = F
print("Patch applied. Safe to import basicsr and realesrgan now.")

Patch applied. Safe to import basicsr and realesrgan now.


# Load All Models

*  RRDBNet = the neural network architecture Real-ESRGAN uses,  RRDB = Residual in Residual Dense Block It's a specific type of network designed for upscaling images, We need to import this to build the model structure before loading weights
*  RealESRGANer = the wrapper class that makes Real-ESRGAN easy to use It handles: loading weights, tiling large images, upscaling, post-processing "tiling" = splitting a large image into small tiles, upscaling each one,then stitching them back together — prevents running out of VRAM
*   BlipProcessor = prepares images before feeding to BLIP model
BlipForConditionalGeneration = the BLIP model that reads images and writes captions


In [3]:
import sys
import torchvision.transforms.functional as F
sys.modules["torchvision.transforms.functional_tensor"] = F
import torch
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer   
from diffusers import StableDiffusionInpaintPipeline
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import numpy as np
import os
import urllib


weights_dir = "/kaggle/working/weights"
os.makedirs(weights_dir, exist_ok=True)
weights = f"{weights_dir}/RealESRGAN_x4plus.pth"

if not os.path.exists(weights):
    url = "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"
    urllib.request.urlretrieve(url, weights)
    
esrga= RRDBNet(
    num_in_ch= 3,
    num_out_ch= 3,
    num_feat= 64,
    num_block= 23,
    num_grow_ch= 32,
    scale= 4
)

enhancer= RealESRGANer(
    scale=4,
    model_path= weights,
    model= esrga,
    tile= 512,# tile=512 means: split the input image into 512×512 tiles
    tile_pad= 10,  # tile_pad=10 = add 10 pixels of overlap between tiles Without overlap, you'd see visible seam lines where tiles meet
    pre_pad=0, # pre_pad = extra padding added before processing
    half= True,  # half=True = use float16 (half precision)
)

print("REAL-ESRGAN Ready")

inpaint = StableDiffusionInpaintPipeline.from_pretrained(
    "runwayml/stable-diffusion-inpainting",
    torch_dtype=torch.float16,
)
inpaint = inpaint.to("cuda")
inpaint.enable_model_cpu_offload()

print("SD Inpainting loaded")

blipprocessor= BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)

blip_model= BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base",
    torch_dtype= torch.float16
).to("cuda")

print("All Models Ready!")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


REAL-ESRGAN Ready


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


All Models Ready!


# Enhancement Function

In [4]:
import gradio as gr

def enhance_image(image, scale_factor): # scale_factor = string from dropdown: "2x" or "4x"
    if Image is None:
        raise gr.Error("Please Upload an Image First..!")

    #convert PIL image to numpy array
    image_array= np.array(image)
    # Real-ESRGAN's enhance() method expects a numpy array, not a PIL image
    # np.array(image) converts PIL → numpy array of shape (height, width, 3)
    # Each pixel is [R, G, B] with values 0-255

    image_array= image_array[:,:,:3]
    # REALESRGAN works only on RGB channels not RGBA(with alpha channel), this slicing helps in keeping first 3 channels only 
    # if a user gives transparent png, this saves from crashing
    outscale= 4 if scale_factor== "4x" else 2

    try:
        output_array, _ = enhancer.enhance(
            image_array,
            outscale= outscale,
        )

    except RuntimeError as e:
        raise gr.Error(f"Enhancement failed: {str(e)}. try smaller image.")

    #convert numpy array to PIL image
    output_image= Image.fromarray(output_array)

    # original and after sizes comparison
    original_size= f"{image.width}x{image.height}"
    new_size= f"{output_image.width}x{output_image.height}"
    size_info= f"Original: {original_size} -> enhanced: {new_size}"
    return output_image, size_info

# Outpainting Function

In [13]:
def get_caption(image):
    inputs= blipprocessor(image, return_tensors= "pt").to("cuda", torch.float16)
    output= blip_model.generate(**inputs, max_new_tokens=60)
    caption= blipprocessor.decode(output[0], skip_special_tokens= True)
    return caption
    
def build_prompt(caption):
    return(
        f"seamless continuation of a scene with {caption}, "
        f"same lighting, same background, same color palette, "
        f"photorealistic, high quality, natural extension,realistic"
    )

def extend_one_side(image, direction, pixels, prompt, negative_prompt):
     # This is the core function — extends the image on ONE side only
     # Called multiple times for multi-direction extension
    orig_w= image.width
    orig_h= image.height

    
    new_w  = orig_w   # default: same width
    new_h  = orig_h   # default: same height
    paste_x = 0           # default: original goes to top-left
    paste_y = 0
    #calculate new canvas size
    if direction== "left":
        new_w= orig_w + pixels
        paste_x= pixels

    elif direction== "right":
        new_w= orig_w + pixels
        paste_x= 0

    elif direction== "top":
        new_h= orig_h + pixels
        paste_y= pixels

    else:
        new_h= orig_h+pixels
        paste_y= 0
    

    new_w= (new_w//8)*8 
    new_h= (new_h//8)* 8
    # Why to convert into multiple of 8? SD's UNet processes images in 8×8 or 64×64 patches internally
    # If dimensions aren't divisible by 8, the math doesn't work out exactly
    
    # Recalculate pixels after rounding to stay consistent
    if direction in ["left", "right"]:
        pixels= new_w- orig_w
    else:
        pixels= new_h- orig_h
    if direction== "left":
        paste_x= pixels
    if direction== "top":
        paste_y= pixels
    #creating black canvas
    canvas= Image.new("RGB", (new_w, new_h), color= (0,0,0))
    # Image.new("RGB", size, color) = create a blank new image
    canvas.paste(image, (paste_x, paste_y))
    # .paste(what_to_paste, where_to_paste_it)
    # (pad_left, pad_top) = the (x, y) pixel coordinate of the top-left corner
    
    
    # ── Resize canvas to 512 preserving aspect ratio ──────────
    sd_size = 512
    ratio   = sd_size / max(new_w, new_h)
    scaled_w= int(new_w*ratio)
    scaled_h= int(new_h* ratio)
    # Round to multiple of 8 for SD
    scaled_w = (scaled_w // 8) * 8
    scaled_h = (scaled_h // 8) * 8
    canvas_sd= canvas.resize((sd_size, sd_size), Image.LANCZOS)

    
    #create the mask
    mask_sd= Image.new("L", (scaled_w, scaled_h), 255)
    from PIL import ImageDraw
    draw= ImageDraw.Draw(mask_sd)
    # Start: everything white (generate everywhere)
    # "L" = grayscale mode
    # The mask is a grayscale image: same size as canvas
    # WHITE (255) = "SD, fill this area"
    # BLACK (0)   = "SD, leave this area exactly as it is"
    feather= 32
    # feather = the overlap zone in pixels between original and generated area

    # Scale the original image boundaries into the new coordinate space
    scale_x= scaled_w/ new_w
    scale_y= scaled_h/new_h
    # px_sd, py_sd = where original starts in the scaled canvas
    # ow_sd, oh_sd = original image dimensions in scaled space
    px_sd= int(paste_x* scale_x)
    py_sd= int(paste_y* scale_y)
    ow_sd    = int(orig_w  * scale_x)
    oh_sd    = int(orig_h  * scale_y)
    

    if direction== "left":
        draw.rectangle([
            px_sd + feather, feather, # top-left of black rect
            scaled_w- feather, scaled_h- feather #bottom left of black rect
        ], fill= 0) #fill=0, black, sd keeps this
    elif direction== "right":
        draw.rectangle([
            feather, feather,
            ow_sd- feather, scaled_h- feather
        ], fill= 0)
    elif direction== "top":
        draw.rectangle([
            feather, py_sd+feather,
            scaled_w- feather, scaled_h- feather
        ], fill=0)
    elif direction== "bottom":
        draw.rectangle([
            feather, feather,
            scaled_w- feather, oh_sd- feather
        ], fill=0)
   
    #run SD inpainting
    result= inpaint(
        prompt= prompt,
        image= canvas_sd,
        mask_image= mask_sd,
        height= sd_size,
        width= sd_size,
        num_inference_steps= 40,
        guidance_scale= 9.0,
        negative_prompt= negative_prompt,
        )
    generated= result.images[0]
    generated_new= generated.resize((new_w, new_h), Image.LANCZOS)
    # Pasting back overrides any changes SD made to that area
    generated_new.paste(image, (paste_x, paste_y))
    return generated_new
    
def outpaint_image(image, direction, extend_percent, custom_prompt, progress=gr.Progress()):
    if image is None:
        raise gr.Error("Please upload an image first...!")
     #resize input
    max_side= 512
    ratio= min(max_side/ image.width, max_side/image.height)
    # Example: 800×600 image, max_side=512
    #   ratio for width:  512/800 = 0.64
    #   ratio for height: 512/600 = 0.853
    #   min(0.64, 0.853) = 0.64 → use 0.64 (shrink based on the wider side)
    # This ensures BOTH dimensions end up ≤ 512
    if ratio< 1.0:
        new_size= (int(image.width*ratio), int(image.height*ratio))
        image= image.resize(new_size, Image.LANCZOS)
        # Image.LANCZOS = high quality resampling algorithm for shrinking
    
    progress(0.1, desc= "Analyzing Image")
    
    # get prompt
    if custom_prompt.strip():
        blip_caption= custom_prompt.strip()
    else:
        blip_caption= get_caption(image)

    prompt= build_prompt(blip_caption)
    progress(0.3, desc= "Preparing Canvas and Mask")
    
    negative_prompt= (
        "blurry, bad quality, ugly, deformed, watermark, text, "
        "duplicate, tiled, repeated pattern, border, frame, "
        "seam, visible edge, color mismatch, different lighting, "
        "different style, distorted"
        )

    # Calculate how many pixels to add
    extend= extend_percent/100.0
    h_pixels= int(image.width * extend)
    # h_pixels = pixels to add on LEFT or RIGHT
    v_pixels= int(image.height * extend)
    # v_pixels = pixels to add on TOP or BOTTOM

    #build the sequences of passes
    if direction== "Horizontal":
        passes= [
            ("right", h_pixels),
            ("left", h_pixels)
        ]
    elif direction== "Vertical":
        passes= [
            ("bottom", v_pixels),
            ("top", v_pixels)
        ]
    else:
        passes=[
            ("bottom", v_pixels),
            ("top", v_pixels),
            ("right", h_pixels),
            ("left", h_pixels)
        ]
    total_passes= len(passes)
    current_image= image
    # current_image starts as the user's upload
    # gets updated after each pass — each pass builds on the previous


    for i, (side, pixels) in enumerate(passes):
        progress_value= 0.1+(0.8* (i/total_passes))
        progress(progress_value, desc= f"Extending {side}... (pass {i+1}/{total_passes})")
        current_image= extend_one_side(
            current_image, side, pixels, prompt, negative_prompt
        )

    progress(1.0, desc= "done!")

    return(
        current_image,
        f"BLIP caption:\n{blip_caption}\n\nFull prompt:\n{prompt}"
    )

# Gradio Interface

In [14]:
with gr.Blocks(
    title= "Image Enhancer & Outpainter"
) as demo:
    gr.Markdown("# Image Enhancer & Outpainter")
    gr.Markdown(
        "**Tab 1** - Enhance: make any image sharper and larger using Real-ESRGAN  \n"
        "**Tab 2** — Outpaint: extend image borders with AI-generated content"
    )
    with gr.Tabs():
        #____________Tab 1- Enhancement____________
        with gr.Tab("Enhance- Super Resolution"):
            gr.Markdown(
                "Upload any image. Real-ESRGAN will sharpen details "
                "and upscale it 2x or 4x."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    enh_input= gr.Image(
                        label= "Upload Image",
                        type= "pil"
                    )
                    enh_scale= gr.Dropdown(
                        choices= ["2x", "4x"],
                        value= "4x",
                        label= "Upscale Factor",
                        info= "4x recommended. Use 2x for very large input images."
                    )
                    enh_btn= gr.Button(
                        "Enhance Image",
                        variant= "Primary"
                    )
                with gr.Column(scale=1):
                    enh_output= gr.Image(
                        label= "Enhanced Image",
                        type= "pil",
                        interactive= False
                    )
                    enh_info= gr.Textbox(
                        label= "Size Info",
                        interactive= False
                    )

            enh_btn.click(
                fn= enhance_image,
                inputs= [enh_input, enh_scale],
                outputs= [enh_output, enh_info]
            )
        #__________Outpaint___________________
        with gr.Tab("Outpaint- Extend Image"):
            gr.Markdown(
                "Upload an image. Choose a direction and how much to extend. "
                "BLIP automatically reads your image to guide the generation — "
                "no text prompt needed."
            )
            with gr.Row():
                with gr.Column(scale=1):
                    out_input= gr.Image(
                        label= "Upload Image to Extend",
                        type= "pil",
                        key= "outpaint_image" #key gives this component a unique name
                    )
                    out_direction= gr.Radio(
                        choices= ["Horizontal", "Vertical", "Both"],
                        value= "Horizontal",
                        label= "Extension Direction",
                        info= (
                            "Horizontal = extend left and right  |  "
                            "Vertical = extend top and bottom  |  "
                            "Both = extend all four sides"
                        )
                        
                    )

                    out_percent= gr.Slider(
                        minimum= 10,
                        maximum= 50,
                        value= 25,
                        step=5,
                        label= "Extend by (%)",
                        info= (
                            "How much to add on each side as % of original size. "
                            "25% means each side grows by a quarter of the original."
                        )
                    )
                    out_prompt= gr.Textbox(
                        label= "Custom Prompt (optional)",
                        placeholder= ("Leave empty to auto-detect with BLIP. "
                            "Or type: 'a forest with tall trees and soft sunlight'"),
                        lines=2
                    )
                    out_btn= gr.Button(
                        "Outpaint Image",
                        variant= "primary"
                    )
                with gr.Column(scale=1):
                    out_output= gr.Image(
                        label= "Extended Result",
                        type= "pil",
                        interactive= False,
                    )
                    out_caption= gr.Textbox(
                        label= "prompt used for generation",
                        interactive= False,
                        lines=3,
                    )
            out_btn.click(
                fn= outpaint_image,
                inputs= [out_input, out_direction, out_percent, out_prompt],
                outputs= [out_output, out_caption],
            )

#_______Launch________
demo.launch(
    share= True,
    debug= True,
    
)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c854fb89d6800cfbd4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 420, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

...

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

  0%|          | 0/40 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c854fb89d6800cfbd4.gradio.live
